In [1]:
# Проверка задания п2.1.2 на устройстве: посылает плате команды занятия, включая проверочную ping,
# и записывает обмен в файл device-2-1-2.log.

import time
from datetime import datetime

import serial
from serial.tools import list_ports

VENDOR_ID = 0x2E8A
PRODUCT_ID = 0x000A

TASK = "2.1.2"
PROJECT = "211-command-usb"
LOG_NAME = "device-2-1-2.log"
COMMANDS = ["enable", "disable", "info", "ping", "nosuchcommand"]
ANSWER_S = 2


def find_board():
    for port in list_ports.comports():
        if port.vid == VENDOR_ID and port.pid == PRODUCT_ID:
            return port
    return None


def talk(board):
    exchange = []
    with serial.Serial(board.device, timeout=0.5) as port:
        time.sleep(0.2)
        port.reset_input_buffer()
        started = time.monotonic()
        for command in COMMANDS:
            port.write((command + "\n").encode("ascii"))
            exchange.append((time.monotonic() - started, "-->", command))
            deadline = time.monotonic() + ANSWER_S
            while time.monotonic() < deadline:
                line = port.readline().decode("ascii", "replace").strip()
                if line:
                    exchange.append((time.monotonic() - started, "<--", line))
                    print(line, end="\r\n")
    return exchange


def write_log(board, exchange):
    with open(LOG_NAME, "w", encoding="utf-8") as log:
        log.write("задание: " + TASK + "\n")
        log.write("проект: " + PROJECT + "\n")
        log.write("устройство: %04x:%04x\n" % (board.vid, board.pid))
        log.write("серийный номер: " + str(board.serial_number) + "\n")
        log.write("порт: " + board.device + "\n")
        log.write("начало: " + datetime.now().isoformat(timespec="seconds") + "\n")
        for moment, direction, text in exchange:
            log.write("%8.3f %s %s\n" % (moment, direction, text))
        answers = len(exchange) - len(COMMANDS)
        log.write("итог: отправлено команд %d, принято строк %d\n" % (len(COMMANDS), answers))


board = find_board()

if board is None:
    print("Плата не найдена. Проверьте кабель и запишите на плату прошивку задания.", end="\r\n")
else:
    print("Плата на порту " + board.device + ", отправляю команды", end="\r\n")
    exchange = talk(board)
    write_log(board, exchange)
    print("Обмен записан в " + LOG_NAME, end="\r\n")


Плата на порту COM3, отправляю команды
enable
dbg read_line:103 got enable
inf cmd_enable:29 led on
disable
dbg read_line:103 got disable
inf cmd_disable:35 led off
info
dbg read_line:103 got info
project: 211-command-usb
repo: https://github.com/dordienkodv/es-student
board: pico
serial: E66118C4E35B702B
chip: manufacturer 0x927, part 0x0002, revision 2
pico-sdk: 2.3.0
ping
dbg read_line:103 got ping
pong
nosuchcommand
dbg read_line:103 got nosuchcommand
err handle_command:84 unknown command: nosuchcommand
Обмен записан в device-2-1-2.log
